In [ ]:
import serial
import time

# 若舊的 Serial 連線仍存在，先關閉
if "ser" in globals() and ser.is_open:
    ser.close()

ser = serial.Serial(
    port="/dev/ttyACM0",
    baudrate=9600,
    timeout=1,
    write_timeout=1
)

time.sleep(3)
ser.reset_input_buffer()

print("Arduino connected successfully!")

Arduino connected successfully!


In [ ]:

import time

# ==========================================
# 讀取 Arduino 左右超音波測距
# ==========================================

def read_all_sensors(ser):

    # 清除先前尚未讀取的資料
    ser.reset_input_buffer()

    # 向 Arduino 要求兩組測距資料
    ser.write(b"GET_ALL\n")
    ser.flush()

    results = {
        "DIST_LEFT": None,
        "DIST_RIGHT": None
    }

    received = set()
    deadline = time.monotonic() + 2

    while time.monotonic() < deadline:

        line = ser.readline().decode(
            "utf-8",
            errors="replace"
        ).strip()

        if ":" not in line:
            continue

        name, value = line.split(":", 1)

        if name not in results:
            continue

        if value == "INVALID":
            results[name] = None

        else:
            try:
                results[name] = float(value)
            except ValueError:
                continue

        received.add(name)

        # 左右兩組資料都收到就結束
        if len(received) == 2:
            break

    return results, received


# ==========================================
# 連續監測左右超音波
# ==========================================

try:
    for i in range(20):

        data, received = read_all_sensors(ser)

        print("Reading", i + 1)

        # 顯示左、右超音波數值
        for name, value in data.items():

            if name not in received:
                print(name, ": NO RESPONSE")

            elif value is None:
                print(name, ": INVALID")

            else:
                print("{}: {:.1f} cm".format(
                    name, value
                ))

        # 取得左右距離
        left = data["DIST_LEFT"]
        right = data["DIST_RIGHT"]

        # ==================================
        # 障礙物距離警示
        # ==================================

        if len(received) < 2:
            print("WARNING: Sensor response incomplete")

        elif left is None or right is None:
            print("WARNING: Sensor data unavailable")

        else:
            nearest = min(left, right)

            print("Nearest obstacle: {:.1f} cm".format(
                nearest
            ))

            if nearest <= 10:
                print("WARNING: Obstacle too close!")

            elif nearest <= 20:
                print("CAUTION: Obstacle nearby")

            else:
                print("Ultrasonic readings received")

        print("---------------------")

        time.sleep(0.5)

except KeyboardInterrupt:
    print("Monitoring stopped")


Reading 1
DIST_LEFT: 40.0 cm
DIST_RIGHT: 29.3 cm
Nearest obstacle: 29.3 cm
Ultrasonic readings received
---------------------
Reading 2
DIST_LEFT: 36.3 cm
DIST_RIGHT: 29.4 cm
Nearest obstacle: 29.4 cm
Ultrasonic readings received
---------------------
Reading 3
DIST_LEFT: 37.2 cm
DIST_RIGHT: 29.4 cm
Nearest obstacle: 29.4 cm
Ultrasonic readings received
---------------------
Reading 4
DIST_LEFT: 41.9 cm
DIST_RIGHT: 29.3 cm
Nearest obstacle: 29.3 cm
Ultrasonic readings received
---------------------
Reading 5
DIST_LEFT: 36.8 cm
DIST_RIGHT: 29.4 cm
Nearest obstacle: 29.4 cm
Ultrasonic readings received
---------------------
Reading 6
DIST_LEFT: 36.8 cm
DIST_RIGHT: 29.4 cm
Nearest obstacle: 29.4 cm
Ultrasonic readings received
---------------------
Reading 7
DIST_LEFT: 41.9 cm
DIST_RIGHT: 29.3 cm
Nearest obstacle: 29.3 cm
Ultrasonic readings received
---------------------
Reading 8
DIST_LEFT: 40.3 cm
DIST_RIGHT: 29.3 cm
Nearest obstacle: 29.3 cm
Ultrasonic readings received
--------------

In [ ]:
import serial
import time

time.sleep(3)
ser.reset_input_buffer()

def send_vision_state(state):
    """
    1 = GREEN  正常跟隨
    2 = YELLOW 停止或放慢
    3 = RED    目標遺失
    """
    if state not in (1, 2, 3):
        raise ValueError("Invalid state")

    ser.write("{}\n".format(state).encode("utf-8"))
    ser.flush()

    response = ser.readline().decode(
        "utf-8", errors="replace"
    ).strip()

    return response


In [ ]:
print(send_vision_state(1))
time.sleep(1)

print(send_vision_state(2))
time.sleep(1)

print(send_vision_state(3))
